# 1B_1. Data Binning
## Opening Data

In [7]:
# Step 1: import packages
import numpy as np
import pandas as pd

In [8]:
data_path = "./32130_2026A_88.csv"
adata = pd.read_csv(data_path)
# adata
adata.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   source ID             3000 non-null   int64  
 1   age                   3000 non-null   int64  
 2   gender                3000 non-null   object 
 3   marital_status        3000 non-null   object 
 4   education_level       3000 non-null   object 
 5   annual_income         3000 non-null   float64
 6   monthly_income        3000 non-null   float64
 7   employment_status     3000 non-null   object 
 8   debt_to_income_ratio  3000 non-null   float64
 9   credit_score          3000 non-null   int64  
 10  loan_amount           3000 non-null   float64
 11  loan_purpose          3000 non-null   object 
 12  interest_rate         3000 non-null   float64
 13  loan_term             3000 non-null   int64  
 14  installment           3000 non-null   float64
 15  num_of_open_accounts 

## debt_to_income_ratio
### Binning


In [9]:
# equi-width cutting
names = ['Bin1', 'Bin2', 'Bin3', 'Bin4', "Bin5"]
bins_num = 5

# Get bin edges
bin_edges = pd.cut(adata['debt_to_income_ratio'], bins=bins_num, retbins=True, labels=names)[1]
# Cut and Get bin labels
adata['debt_to_income_ratio_EWBinNum(n=5)'] = pd.cut(adata['debt_to_income_ratio'], bins=bins_num, labels=names)
# Map each bin label to its range
bin_ranges = {names[i]: f"[{bin_edges[i]}, {bin_edges[i+1]})" for i in range(len(names))}
adata['bin_range(debt_to_income_ratio)'] = adata['debt_to_income_ratio_EWBinNum(n=5)'].map(bin_ranges)

# Print
print(adata[['debt_to_income_ratio', 'debt_to_income_ratio_EWBinNum(n=5)', 'bin_range(debt_to_income_ratio)']].sort_values('debt_to_income_ratio'))
print(adata[['debt_to_income_ratio', 'debt_to_income_ratio_EWBinNum(n=5)']].sort_values('debt_to_income_ratio').groupby('debt_to_income_ratio_EWBinNum(n=5)', observed=True).count())

      debt_to_income_ratio debt_to_income_ratio_EWBinNum(n=5)  \
1381                 0.010                               Bin1   
2517                 0.010                               Bin1   
2391                 0.010                               Bin1   
2649                 0.010                               Bin1   
2554                 0.010                               Bin1   
...                    ...                                ...   
845                  0.517                               Bin5   
796                  0.527                               Bin5   
529                  0.550                               Bin5   
2828                 0.562                               Bin5   
2911                 0.570                               Bin5   

     bin_range(debt_to_income_ratio)  
1381  [0.00944, 0.12199999999999998)  
2517  [0.00944, 0.12199999999999998)  
2391  [0.00944, 0.12199999999999998)  
2649  [0.00944, 0.12199999999999998)  
2554  [0.00944, 0.121999

In [10]:
# equi-depth cutting
names = ['Bin1', 'Bin2', 'Bin3', 'Bin4', 'Bin5']
bins_num = 5

# Cut and get bin labels
adata['debt_to_income_ratio_EDBinNum(n=5)'] = pd.qcut(adata['debt_to_income_ratio'], q = bins_num, labels=names)

print(adata[['debt_to_income_ratio', 'debt_to_income_ratio_EDBinNum(n=5)']].sort_values('debt_to_income_ratio'))
print(adata[['debt_to_income_ratio', 'debt_to_income_ratio_EDBinNum(n=5)']].sort_values('debt_to_income_ratio').groupby('debt_to_income_ratio_EDBinNum(n=5)', observed=True).count())

      debt_to_income_ratio debt_to_income_ratio_EDBinNum(n=5)
1381                 0.010                               Bin1
2517                 0.010                               Bin1
2391                 0.010                               Bin1
2649                 0.010                               Bin1
2554                 0.010                               Bin1
...                    ...                                ...
845                  0.517                               Bin5
796                  0.527                               Bin5
529                  0.550                               Bin5
2828                 0.562                               Bin5
2911                 0.570                               Bin5

[3000 rows x 2 columns]
                                    debt_to_income_ratio
debt_to_income_ratio_EDBinNum(n=5)                      
Bin1                                                 610
Bin2                                                 603
Bin

## Smoothing

In [11]:
# smoothing with means of Eq Width bins

# Calculate means
bin_mean = adata[['debt_to_income_ratio', 'debt_to_income_ratio_EWBinNum(n=5)']].groupby('debt_to_income_ratio_EWBinNum(n=5)', observed=True).mean()

# Merge on bins
adata = adata.merge(bin_mean, on = 'debt_to_income_ratio_EWBinNum(n=5)', suffixes=('','_smoothed_EWBin'))
print(adata[['debt_to_income_ratio', 'debt_to_income_ratio_EWBinNum(n=5)', 'debt_to_income_ratio_smoothed_EWBin']].sort_values('debt_to_income_ratio'))

      debt_to_income_ratio debt_to_income_ratio_EWBinNum(n=5)  \
1381                 0.010                               Bin1   
2517                 0.010                               Bin1   
2391                 0.010                               Bin1   
2649                 0.010                               Bin1   
2554                 0.010                               Bin1   
...                    ...                                ...   
845                  0.517                               Bin5   
796                  0.527                               Bin5   
529                  0.550                               Bin5   
2828                 0.562                               Bin5   
2911                 0.570                               Bin5   

      debt_to_income_ratio_smoothed_EWBin  
1381                             0.073631  
2517                             0.073631  
2391                             0.073631  
2649                             0.073631  

In [12]:
# smoothing with means of Eq Didth bins

# Calculate means
bin_mean = adata[['debt_to_income_ratio', 'debt_to_income_ratio_EDBinNum(n=5)']].groupby('debt_to_income_ratio_EDBinNum(n=5)', observed=True).mean()

# Merge on bins
adata = adata.merge(bin_mean, on = 'debt_to_income_ratio_EDBinNum(n=5)', suffixes=('','_smoothed_EDBin'))
print(adata[['debt_to_income_ratio', 'debt_to_income_ratio_EDBinNum(n=5)', 'debt_to_income_ratio_smoothed_EDBin']].sort_values('debt_to_income_ratio'))

      debt_to_income_ratio debt_to_income_ratio_EDBinNum(n=5)  \
1381                 0.010                               Bin1   
2517                 0.010                               Bin1   
2391                 0.010                               Bin1   
2649                 0.010                               Bin1   
2554                 0.010                               Bin1   
...                    ...                                ...   
845                  0.517                               Bin5   
796                  0.527                               Bin5   
529                  0.550                               Bin5   
2828                 0.562                               Bin5   
2911                 0.570                               Bin5   

      debt_to_income_ratio_smoothed_EDBin  
1381                             0.052477  
2517                             0.052477  
2391                             0.052477  
2649                             0.052477  

## credit_score
### bunning

In [13]:
# equi-width cutting
names = ['Bin1', 'Bin2', 'Bin3', 'Bin4', "Bin5"]
bins_num = 5

# Get bin edges
bin_edges_credit_score = pd.cut(adata['credit_score'], bins=bins_num, retbins=True, labels=names)[1]
# Cut and Get bin labels
adata['credit_score_EWBinNum(n=5)'] = pd.cut(adata['credit_score'], bins=bins_num, labels=names)
# Map each bin label to its range
bin_ranges = {names[i]: f"[{bin_edges_credit_score[i]}, {bin_edges_credit_score[i+1]})" for i in range(len(names))}
adata['bin_range (credit_score)'] = adata['credit_score_EWBinNum(n=5)'].map(bin_ranges)

# Print
print(adata[['credit_score', 'credit_score_EWBinNum(n=5)', 'bin_range (credit_score)']].sort_values('credit_score'))
print(adata[['credit_score', 'credit_score_EWBinNum(n=5)']].sort_values('credit_score').groupby('credit_score_EWBinNum(n=5)', observed=True).count())

      credit_score credit_score_EWBinNum(n=5) bin_range (credit_score)
27             450                       Bin1           [449.6, 530.0)
1409           461                       Bin1           [449.6, 530.0)
2472           470                       Bin1           [449.6, 530.0)
1570           477                       Bin1           [449.6, 530.0)
1037           478                       Bin1           [449.6, 530.0)
...            ...                        ...                      ...
959            850                       Bin5           [770.0, 850.0)
731            850                       Bin5           [770.0, 850.0)
715            850                       Bin5           [770.0, 850.0)
2996           850                       Bin5           [770.0, 850.0)
54             850                       Bin5           [770.0, 850.0)

[3000 rows x 3 columns]
                            credit_score
credit_score_EWBinNum(n=5)              
Bin1                                  49


In [14]:
# equi-depth cutting
names = ['Bin1', 'Bin2', 'Bin3', 'Bin4', 'Bin5']
bins_num = 5

# Cut and get bin labels
adata['credit_score_EDBinNum(n=5)'] = pd.qcut(adata['credit_score'], q = bins_num, labels=names)

print(adata[['credit_score', 'credit_score_EDBinNum(n=5)']].sort_values('credit_score'))
print(adata[['credit_score', 'credit_score_EDBinNum(n=5)']].sort_values('credit_score').groupby('credit_score_EDBinNum(n=5)', observed=True).count())

      credit_score credit_score_EDBinNum(n=5)
27             450                       Bin1
1409           461                       Bin1
2472           470                       Bin1
1570           477                       Bin1
1037           478                       Bin1
...            ...                        ...
959            850                       Bin5
731            850                       Bin5
715            850                       Bin5
2996           850                       Bin5
54             850                       Bin5

[3000 rows x 2 columns]
                            credit_score
credit_score_EDBinNum(n=5)              
Bin1                                 604
Bin2                                 601
Bin3                                 613
Bin4                                 585
Bin5                                 597


### Smoothing

In [15]:
# smoothing with means of Eq Width bins

# Calculate means
bin_mean = adata[['credit_score', 'credit_score_EWBinNum(n=5)']].groupby('credit_score_EWBinNum(n=5)', observed=True).mean()

# Merge on bins
adata = adata.merge(bin_mean, on = 'credit_score_EWBinNum(n=5)', suffixes=('','_smoothed_EWBin'))
print(adata[['credit_score', 'credit_score_EWBinNum(n=5)', 'credit_score_smoothed_EWBin']].sort_values('credit_score'))

      credit_score credit_score_EWBinNum(n=5)  credit_score_smoothed_EWBin
27             450                       Bin1                   504.734694
1409           461                       Bin1                   504.734694
2472           470                       Bin1                   504.734694
1570           477                       Bin1                   504.734694
1037           478                       Bin1                   504.734694
...            ...                        ...                          ...
959            850                       Bin5                   802.029605
731            850                       Bin5                   802.029605
715            850                       Bin5                   802.029605
2996           850                       Bin5                   802.029605
54             850                       Bin5                   802.029605

[3000 rows x 3 columns]


In [16]:
# smoothing with means of Eq Depth bins

# Calculate means
bin_mean = adata[['credit_score', 'credit_score_EDBinNum(n=5)']].groupby('credit_score_EDBinNum(n=5)', observed=True).mean()

# Merge on bins
adata = adata.merge(bin_mean, on = 'credit_score_EDBinNum(n=5)', suffixes=('','_smoothed_EDBin'))
print(adata[['credit_score', 'credit_score_EDBinNum(n=5)', 'credit_score_smoothed_EDBin']].sort_values('credit_score'))

      credit_score credit_score_EDBinNum(n=5)  credit_score_smoothed_EDBin
27             450                       Bin1                   581.324503
1409           461                       Bin1                   581.324503
2472           470                       Bin1                   581.324503
1570           477                       Bin1                   581.324503
1037           478                       Bin1                   581.324503
...            ...                        ...                          ...
959            850                       Bin5                   779.410385
731            850                       Bin5                   779.410385
715            850                       Bin5                   779.410385
2996           850                       Bin5                   779.410385
54             850                       Bin5                   779.410385

[3000 rows x 3 columns]


### Export CSV

In [17]:
# create new var final and put result in it
final = adata[['source ID', 'debt_to_income_ratio', 'debt_to_income_ratio_EWBinNum(n=5)', 'debt_to_income_ratio_EDBinNum(n=5)', 'bin_range(debt_to_income_ratio)', 'debt_to_income_ratio_smoothed_EDBin', 'debt_to_income_ratio_smoothed_EWBin', 'credit_score', 'credit_score_EWBinNum(n=5)', 'credit_score_EDBinNum(n=5)', 'bin_range (credit_score)', 'credit_score_smoothed_EDBin', 'credit_score_smoothed_EWBin']].sort_values('debt_to_income_ratio')

# round to 3 decimal places
final['debt_to_income_ratio_smoothed_EDBin'] = final['debt_to_income_ratio_smoothed_EDBin'].round(3)
final['debt_to_income_ratio_smoothed_EWBin'] = final['debt_to_income_ratio_smoothed_EWBin'].round(3)
final['credit_score_smoothed_EDBin'] = final['credit_score_smoothed_EDBin'].round(3)
final['credit_score_smoothed_EWBin'] = final['credit_score_smoothed_EWBin'].round(3)


final.to_csv('1B_1_Binning.csv', index=False)
final

,source ID,debt_to_income_ratio,debt_to_income_ratio_EWBinNum(n=5),debt_to_income_ratio_EDBinNum(n=5),bin_range(debt_to_income_ratio),debt_to_income_ratio_smoothed_EDBin,debt_to_income_ratio_smoothed_EWBin,credit_score,credit_score_EWBinNum(n=5),credit_score_EDBinNum(n=5),bin_range (credit_score),credit_score_smoothed_EDBin,credit_score_smoothed_EWBin
1381,15718,0.010,Bin1,Bin1,"[0.00944, 0.12199999999999998)",0.052,0.074,738,Bin4,Bin4,"[690.0, 770.0)",721.508,726.676
2517,17978,0.010,Bin1,Bin1,"[0.00944, 0.12199999999999998)",0.052,0.074,615,Bin3,Bin1,"[610.0, 690.0)",581.325,654.629
2391,111,0.010,Bin1,Bin1,"[0.00944, 0.12199999999999998)",0.052,0.074,631,Bin3,Bin2,"[610.0, 690.0)",645.363,654.629
2649,19758,0.010,Bin1,Bin1,"[0.00944, 0.12199999999999998)",0.052,0.074,557,Bin2,Bin1,"[530.0, 610.0)",581.325,580.636
2554,18301,0.010,Bin1,Bin1,"[0.00944, 0.12199999999999998)",0.052,0.074,586,Bin2,Bin1,"[530.0, 610.0)",581.325,580.636
...,...,...,...,...,...,...,...,...,...,...,...,...,...
845,3950,0.517,Bin5,Bin5,"[0.45799999999999996, 0.57)",0.338,0.489,524,Bin1,Bin1,"[449.6, 530.0)",581.325,504.735
796,17440,0.527,Bin5,Bin5,"[0.45799999999999996, 0.57)",0.338,0.489,727,Bin4,Bin4,"[690.0, 770.0)",721.508,726.676
529,1172,0.550,Bin5,Bin5,"[0.45799999999999996, 0.57)",0.338,0.489,645,Bin3,Bin2,"[610.0, 690.0)",645.363,654.629
2828,32,0.562,Bin5,Bin5,"[0.45799999999999996, 0.57)",0.338,0.489,818,Bin5,Bin5,"[770.0, 850.0)",779.410,802.030


Equi-width Binning: First, using `panda.cut(data[%ColumnName%], bins=5, labels= ['Bin1', 'Bin2', 'Bin3', 'Bin4', "Bin5"])` to divide data into 10 equal-width intervals and store them in new column `%ColumnName%_EWBinNum(n=5)`. The bin range for each group was calculated and documented in the `bin_range(%ColumnName%)` column.

Equi-depth Binning: First, using `pandas.qcut(data[%ColumnName%], q=5, labels=['Bin1', 'Bin2', 'Bin3', 'Bin4', "Bin5"])`, the data was divided into 5 intervals, ensuring an equal number of observations within each group. Then store bin numbers into new column “`%ColumnName%_EDBinNum(n=5)`”

Smoothing: First, use `data[['%ColumnName%', '%ColumnName%_EDBinNum(n=5)']].groupby('%ColumnName%_EDBinNum(n=5)', observed=True).mean()` to calculated means, then use `adata.merge(bin_mean, on = ('%ColumnName%__EDBinNum(n=5)', suffixes=('','_smoothed_EDBin')))` to merge means on bins. For Equi-width and Equi-depth, bin means were stored in “`%ColumnName%_smoothed_EWBin`” and “`%ColumnName%_smoothed_EDBin`”